# Understanding the data and experimenting with collection of external data

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns

## Import and view data

In [4]:
cross_border_payments = pd.read_csv("../data/cross_border_payments.csv")
trade_finance = pd.read_csv("../data/trade_finance.csv")
transactional_banking = pd.read_csv("../data/transactional_banking.csv")

In [7]:
cross_border_payments.head()

,transaction_id,entity_id,entity_name,sector,date,direction,currency_pair,value_zar,counterparty_country,corridor_type,beneficiary_name,reference,memo
0,XBP63220455,E01,BHP Group,mining,2023-07-01,inbound,USD/ZAR,2541553.55,Switzerland,intercompany,BHP Group Switzerland Ltd,INTERCO-730855,NaN
1,XBP14207725,E11,Pepkor Holdings,consumer,2023-07-01,outbound,USD/ZAR,407839.87,Angola,intercompany,Pepkor Holdings Angola Ltd,INTERCO-827118,NaN
2,XBP66460952,E11,Pepkor Holdings,consumer,2023-07-01,outbound,CNY/ZAR,72148.47,Angola,intercompany,Pepkor Holdings Angola Ltd,INTERCO-488519,NaN
3,XBP45973312,E11,Pepkor Holdings,consumer,2023-07-01,outbound,GBP/ZAR,53285.65,Namibia,intercompany,Pepkor Holdings Namibia Ltd,INTERCO-585129,NaN
4,XBP13173829,E11,Pepkor Holdings,consumer,2023-07-01,inbound,AED/ZAR,2858193.91,Japan,trade,Continental Resources Trading,TRADE-568825,NaN


In [8]:
trade_finance.head()

,instrument_id,entity_id,entity_name,sector,date,instrument_type,direction,tenor_days,value_zar,counterparty_country,commodity_or_contract_type,status,beneficiary_name,reference,memo
0,TF67401938,E01,BHP Group,mining,2023-07-01,letters_of_credit,export,60,13499920.55,United Arab Emirates,agri_produce,issued,Silverline Trading Co.,LC-471415,NaN
1,TF91455580,E01,BHP Group,mining,2023-07-01,letters_of_credit,export,365,558304.97,Switzerland,iron_ore,settled,Global Commodities Marketing,LC-266865,NaN
2,TF31370953,E10,Bid Corporation,consumer,2023-07-01,letters_of_credit,export,30,4042335.97,United Kingdom,agri_produce,settled,Pacific International Trading House,LC-946978,NaN
3,TF13634137,E10,Bid Corporation,consumer,2023-07-01,letters_of_credit,import,365,171042.89,China,platinum_group_metals,active,Silverline Resources Trading,LC-553503,NaN
4,TF86438695,E10,Bid Corporation,consumer,2023-07-01,letters_of_credit,export,120,331531.78,Netherlands,electronics,settled,Silverline Resources Trading,LC-260628,NaN


In [9]:
transactional_banking.head()

,transaction_id,entity_id,entity_name,sector,date,leg_type,direction,amount_zar,currency,channel,beneficiary_name,reference,memo
0,TXN40610803,E01,BHP Group,mining,2023-07-01,collections,inbound,63878.473869,ZAR,EFT,Continental Metals Trading House,INV-662227,NaN
1,TXN12547643,E11,Pepkor Holdings,consumer,2023-07-01,supplier_payments,outbound,53220.970000,ZAR,EFT,Sunrise Cold Chain Logistics,INV-591371,NaN
2,TXN37710224,E11,Pepkor Holdings,consumer,2023-07-01,supplier_payments,outbound,12309.970000,ZAR,SWIFT,Sunrise Cold Chain Logistics,PO-687736,NaN
3,TXN65618575,E11,Pepkor Holdings,consumer,2023-07-01,supplier_payments,outbound,6308.300000,ZAR,Internal Transfer,Sunrise Cold Chain Logistics,INV-139304,NaN
4,TXN50222056,E11,Pepkor Holdings,consumer,2023-07-01,supplier_payments,outbound,55346.750000,ZAR,Internal Transfer,Cape Wholesale Distributors,INV-556985,NaN


In [32]:
list(set(transactional_banking.entity_name.unique()) | \
    set(cross_border_payments.entity_name.unique()) | \
    set(trade_finance.entity_name.unique()))

['Sanlam',
 'Vodacom Group',
 'OUTsurance Group',
 'Valterra Platinum',
 'Naspers',
 'Shaftesbury Capital plc',
 'NEPI Rockcastle',
 'Gold Fields',
 'BHP Group',
 'Glencore',
 'AngloGold Ashanti',
 'Shoprite Holdings',
 'Clicks Group',
 'Prosus',
 'Bid Corporation',
 'Pepkor Holdings',
 'Anglo American',
 'Aspen Pharmacare',
 'The Bidvest Group',
 'MTN Group']

Interesting, we only have 20 companies...

#TODO: Double check
My understanding of the data is as follows:
- Transactional: Actual payments made by companies
- Cross border payments: Message sent between banks, used when money is transfered internationally 
(e.g. Syn bank sends a swift message to Barclays that Pepkor transfered 1000 pounds to jane street)
- Trade Finance: Basically communication to ensure that buyers and sellers are protected. It'll 
store things like company a is buying product from company b, then a credit note is reached out to 
tell company b that it will get payed once the product is shipped.

**This means that the same transaction can appear in all 3 datasets, so we need to figure out how 
to remove the same transaction**

**We need to find out what the core product pillars are for wich we are predicting wallet share**

## Retrieving external data

We need to get external data to actually predict wallet size

The goal of this section is not to retrieve data, but more to experiment on possible ways to get 
the data. Once methods are finalized, a Python script will be written to retrieve external data.

In [4]:
# Tickers string for companies, used with yahoo finance
tickers_str = "SLM.JO VOD.JO OUT.JO VAL.JO NPN.JO SHC.JO NRP.JO GFI.JO BHG.JO GLN.JO ANG.JO SHP.JO " \
"CLS.JO PRX.JO BVT.JO PPH.JO AGL.JO APN.JO BTI.JO MTN.JO"

#Used for getting ticker data
company_tickers = {
    "Sanlam": "SLM.JO",
    "Vodacom Group": "VOD.JO",
    "OUTsurance Group": "OUT.JO",
    "Valterra Platinum": "VAL.JO",
    "Naspers": "NPN.JO",
    "Shaftesbury Capital plc": "SHC.JO",
    "NEPI Rockcastle": "NRP.JO",
    "Gold Fields": "GFI.JO",
    "BHP Group": "BHG.JO",
    "Glencore": "GLN.JO",
    "AngloGold Ashanti": "ANG.JO",
    "Shoprite Holdings": "SHP.JO",
    "Clicks Group": "CLS.JO",
    "Prosus": "PRX.JO",
    "Bid Corporation": "BVT.JO",
    "Pepkor Holdings": "PPH.JO",
    "Anglo American": "AGL.JO",
    "Aspen Pharmacare": "APN.JO",
    "The Bidvest Group": "BTI.JO",
    "MTN Group": "MTN.JO",
}

In [2]:
import yfinance as yf

portfolio = yf.Tickers(tickers_str)

In [9]:
dir(portfolio.tickers[company_tickers["Sanlam"]])

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_analysis',
 '_data',
 '_download_options',
 '_earnings',
 '_earnings_dates',
 '_expirations',
 '_fast_info',
 '_fetch_ticker_tz',
 '_financials',
 '_fundamentals',
 '_funds_data',
 '_get_earnings_dates_using_scrape',
 '_get_earnings_dates_using_screener',
 '_get_ticker_tz',
 '_holders',
 '_isin',
 '_lazy_load_price_history',
 '_message_handler',
 '_news',
 '_options2df',
 '_price_history',
 '_quote',
 '_shares',
 '_tz',
 '_underlying',
 'actions',
 'analyst_price_targets',
 'balance_sheet',
 'balancesheet',
 'calendar',
 'capital_gains',
 'cash_flow',
 'cashflow',
 'dividends',
 'earnings',
 'earnings_d